In [11]:
import pandas as pd
from lifelines import CoxPHFitter
# Load patient embeddings and survival data
emb = pd.read_csv("patient_embeddings_with_labels.csv")  # cruk_id + embedding columns
tx = pd.read_csv("data/tracerX.csv")
tx.columns = tx.columns.str.strip()

# Keep essential survival columns
tx_keep = tx[['cruk_id', 'dfs_time', 'cens_dfs']].copy()

# Merge embeddings with survival
df_features = emb.merge(tx_keep, on='cruk_id', how='inner')

# Basic cleaning
df_features = df_features.dropna(subset=['dfs_time', 'cens_dfs'])
df_features = df_features[df_features['dfs_time'] > 0]

# Features: embedding columns
X = df_features[[str(i) for i in range(32)]].values  # shape: (n_samples, 32)

# Labels / survival info
y_time = df_features['dfs_time'].values
y_event = df_features['cens_dfs'].values

# Features: embedding columns
embedding_cols = [str(i) for i in range(32)]
X = df_features[embedding_cols]

# Combine with survival data for lifelines
cox_df = df_features[['dfs_time', 'cens_dfs'] + embedding_cols]

# Split train/test
from sklearn.model_selection import train_test_split
train_df, test_df = train_test_split(cox_df, test_size=0.2, random_state=42)

# Fit Cox model with L2 regularization (alpha)
cph = CoxPHFitter(penalizer=0.1)  # try penalizer=0.1, 0.5, 1.0
cph.fit(train_df, duration_col='dfs_time', event_col='cens_dfs')
cph.print_summary()

# Evaluate on test set
from lifelines.utils import concordance_index
# Predicted risk scores
risk_scores = cph.predict_partial_hazard(test_df)

# Evaluate on test set
risk_scores = cph.predict_partial_hazard(test_df)
c_index = concordance_index(test_df['dfs_time'], -risk_scores, test_df['cens_dfs'])
print(f"Test C-index: {c_index:.4f}")


<lifelines.CoxPHFitter: fitted with 271 total observations, 123 right-censored observations>
             duration col = 'dfs_time'
                event col = 'cens_dfs'
                penalizer = 0.1
                 l1 ratio = 0.0
      baseline estimation = breslow
   number of observations = 271
number of events observed = 148
   partial log-likelihood = -765.53
         time fit was run = 2025-08-13 18:52:12 UTC

---
           coef exp(coef)  se(coef)  coef lower 95%  coef upper 95% exp(coef) lower 95% exp(coef) upper 95%
covariate                                                                                                  
0         -0.28      0.75      1.29           -2.81            2.25                0.06                9.46
1          1.75      5.78      2.72           -3.57            7.08                0.03             1183.95
2         -0.67      0.51      2.76           -6.08            4.75                0.00              115.35
3          0.63      1.87      2.43           -4.14            5.39                0.02              218.77
4         -1.05      0.35      2.29           -5.55            3.44                0.00               31.29
5         -2.16      0.12      2.44           -6.94            2.62                0.00               13.71
6          0.45      1.57      2.74           -4.92            5.83                0.01              340.25
7          0.37      1.44      2.89           -5.29            6.03                0.01              414.71
8         -1.22      0.30      4.73          -10.50            8.06                0.00             3161.77
9         -2.56      0.08      2.66           -7.78            2.66                0.00               14.33
10         0.84      2.31      1.52           -2.14            3.81                0.12               45.24
11         2.07      7.95      3.89           -5.54            9.69                0.00            16150.97
12        -0.14      0.87      1.87           -3.81            3.52                0.02               33.91
13        -0.22      0.81      2.25           -4.63            4.20                0.01               66.52
14        -0.11      0.90      2.29           -4.59            4.37                0.01               79.14
15        -0.32      0.73      2.66           -5.53            4.89                0.00              132.35
16         2.13      8.45      2.48           -2.74            7.00                0.06             1099.30
17         0.41      1.51      2.19           -3.89            4.71                0.02              111.39
18         1.20      3.33      2.03           -2.78            5.19                0.06              179.52
19        -0.09      0.91      1.93           -3.88            3.70                0.02               40.45
20         1.47      4.33      3.07           -4.55            7.48                0.01             1772.47
21        -0.34      0.71      1.79           -3.84            3.17                0.02               23.73
22        -0.46      0.63      2.52           -5.39            4.47                0.00               87.14
23         0.57      1.78      1.92           -3.19            4.34                0.04               76.73
24         0.61      1.84      2.73           -4.75            5.97                0.01              390.63
25        -0.05      0.95      2.51           -4.96            4.86                0.01              129.44
26        -1.55      0.21      2.08           -5.64            2.53                0.00               12.53
27        -0.22      0.80      3.32           -6.72            6.28                0.00              534.18
28        -0.79      0.45      2.16           -5.02            3.44                0.01               31.17
29         0.66      1.94      1.63           -2.53            3.85                0.08               46.81
30         2.29      9.86      4.16           -5.87           10.45                0.00            34465.06
31      

Test C-index: 0.4356


In [12]:
from lifelines import CoxPHFitter
from sklearn.model_selection import train_test_split
from lifelines.utils import concordance_index

# Split train/test
train_df, test_df = train_test_split(cox_df, test_size=0.2, random_state=42)

# Cox model with elastic net regularization
# penalizer: overall strength of regularization
# l1_ratio: 0 -> pure L2, 1 -> pure L1, 0.5 -> mix
cph = CoxPHFitter(penalizer=0.1, l1_ratio=0.5)
cph.fit(train_df, duration_col='dfs_time', event_col='cens_dfs')
cph.print_summary()

# Evaluate on test set
risk_scores = cph.predict_partial_hazard(test_df)
c_index = concordance_index(test_df['dfs_time'], -risk_scores, test_df['cens_dfs'])
print(f"Test C-index: {c_index:.4f}")

<lifelines.CoxPHFitter: fitted with 271 total observations, 123 right-censored observations>
             duration col = 'dfs_time'
                event col = 'cens_dfs'
                penalizer = 0.1
                 l1 ratio = 0.5
      baseline estimation = breslow
   number of observations = 271
number of events observed = 148
   partial log-likelihood = -769.44
         time fit was run = 2025-08-13 18:53:49 UTC

---
           coef exp(coef)  se(coef)  coef lower 95%  coef upper 95% exp(coef) lower 95% exp(coef) upper 95%
covariate                                                                                                  
0          0.00      1.00      0.00           -0.00            0.00                1.00                1.00
1          0.00      1.00      0.00           -0.00            0.00                1.00                1.00
2         -0.00      1.00      0.00           -0.00            0.00                1.00                1.00
3          0.00      1.00      0.00           -0.00            0.00                1.00                1.00
4          0.00      1.00      0.00           -0.00            0.00                1.00                1.00
5         -0.00      1.00      0.00           -0.00            0.00                1.00                1.00
6          0.00      1.00      0.00           -0.00            0.00                1.00                1.00
7          0.00      1.00      0.00           -0.00            0.00                1.00                1.00
8         -0.00      1.00      0.00           -0.00            0.00                1.00                1.00
9         -0.00      1.00      0.00           -0.00            0.00                1.00                1.00
10         0.00      1.00      0.00           -0.00            0.00                1.00                1.00
11         0.00      1.00      0.00           -0.00            0.00                1.00                1.00
12         0.00      1.00      0.00           -0.00            0.00                1.00                1.00
13         0.00      1.00      0.00           -0.00            0.00                1.00                1.00
14         0.00      1.00      0.00           -0.00            0.00                1.00                1.00
15         0.00      1.00      0.00           -0.00            0.00                1.00                1.00
16         0.33      1.39      1.31           -2.23            2.90                0.11               18.15
17        -0.00      1.00      0.00           -0.00            0.00                1.00                1.00
18         0.00      1.00      0.00           -0.00            0.00                1.00                1.00
19         0.00      1.00      0.00           -0.00            0.00                1.00                1.00
20         0.00      1.00      0.00           -0.00            0.00                1.00                1.00
21        -0.00      1.00      0.00           -0.00            0.00                1.00                1.00
22         0.00      1.00      0.00           -0.00            0.00                1.00                1.00
23         0.00      1.00      0.00           -0.00            0.00                1.00                1.00
24         0.00      1.00      0.00           -0.01            0.01                0.99                1.01
25        -0.00      1.00      0.00           -0.00            0.00                1.00                1.00
26         0.00      1.00      0.00           -0.00            0.00                1.00                1.00
27         0.00      1.00      0.00           -0.00            0.00                1.00                1.00
28         0.00      1.00      0.00           -0.00            0.00                1.00                1.00
29         0.00      1.00      0.00           -0.00            0.00                1.00                1.00
30         0.00      1.00      0.00           -0.00            0.00                1.00                1.00
31      

Test C-index: 0.4145
